In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, recall_score, f1_score
from scipy.stats.mstats import winsorize
from imblearn.over_sampling import SMOTE
import os
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [2]:
notebook_dir = os.getcwd()
path = os.path.join(notebook_dir, '..', 'data', 'processed', 'stroke_processed.csv')
stroke_dataset = pd.read_csv(path)

# Drop patient ID - it's a unique identifier, not a predictive feature
stroke_dataset = stroke_dataset.drop(columns=['id'])

In [3]:
print(f'Shape of dataset: {stroke_dataset.shape}')
print(stroke_dataset.dtypes)
stroke_dataset.sample(10)

Shape of dataset: (4254, 11)
gender                object
age                  float64
hypertension            bool
heart_disease           bool
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                  bool
dtype: object


,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
856,Female,27.0,False,False,No,Private,Urban,58.39,30.4,never smoked,False
3883,Female,57.0,False,False,Yes,Private,Urban,83.14,31.9,never smoked,False
3569,Male,25.0,False,False,No,Private,Rural,229.94,23.5,never smoked,False
3527,Female,31.0,False,False,Yes,Private,Rural,69.72,39.5,smokes,False
2807,Female,49.0,False,False,Yes,Private,Rural,73.48,33.0,never smoked,False
1323,Female,54.0,False,False,Yes,Private,Urban,207.79,38.6,never smoked,False
3806,Female,22.0,False,False,No,Private,Rural,102.00,40.4,smokes,False
723,Female,51.0,False,False,No,Govt_job,Rural,116.14,20.9,never smoked,False
1491,Female,22.0,False,False,No,Private,Urban,56.84,29.9,smokes,False
700,Female,51.0,False,False,Yes,Private,Urban,82.59,26.2,formerly smoked,False


In [4]:
stroke_dataset['gender'] = stroke_dataset['gender'].astype('category')
stroke_dataset['ever_married'] = stroke_dataset['ever_married'].astype('category')
stroke_dataset['work_type'] = stroke_dataset['work_type'].astype('category')
stroke_dataset['Residence_type'] = stroke_dataset['Residence_type'].astype('category')
stroke_dataset['smoking_status'] = stroke_dataset['smoking_status'].astype('category')
stroke_dataset['stroke'] = stroke_dataset['stroke'].astype('bool')

In [5]:
stroke_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4254 entries, 0 to 4253
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   gender             4254 non-null   category
 1   age                4254 non-null   float64 
 2   hypertension       4254 non-null   bool    
 3   heart_disease      4254 non-null   bool    
 4   ever_married       4254 non-null   category
 5   work_type          4254 non-null   category
 6   Residence_type     4254 non-null   category
 7   avg_glucose_level  4254 non-null   float64 
 8   bmi                4254 non-null   float64 
 9   smoking_status     4254 non-null   category
 10  stroke             4254 non-null   bool    
dtypes: bool(3), category(5), float64(3)
memory usage: 133.8 KB


In [6]:
print(stroke_dataset['stroke'].value_counts())
print(stroke_dataset['stroke'].value_counts(normalize=True))

stroke
False    4007
True      247
Name: count, dtype: int64
stroke
False    0.941937
True     0.058063
Name: proportion, dtype: float64


In [7]:
for column in ['avg_glucose_level', 'bmi']:
    stroke_dataset[column] = np.log1p(stroke_dataset[column])

## From chi-square test I can eliminate residence_type since it has no relationship to the target variable domain knowledge wise and test wise.

In [8]:
stroke_dataset = stroke_dataset.drop(columns=['Residence_type'])

In [9]:
X = stroke_dataset.drop('stroke', axis=1)
y = stroke_dataset['stroke']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
# Separate categorical and numerical columns
categorical_cols = ['gender', 'ever_married', 'work_type', 'smoking_status']
numerical_cols = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']

# One-hot encode categorical variables
encoder = OneHotEncoder(drop='first', sparse_output=False)
X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

# Get feature names for one-hot encoded columns
cat_feature_names = encoder.get_feature_names_out(categorical_cols)

# Convert to DataFrames
X_train_cat_df = pd.DataFrame(X_train_cat, columns=cat_feature_names, index=X_train.index)
X_test_cat_df = pd.DataFrame(X_test_cat, columns=cat_feature_names, index=X_test.index)

# Combine with numerical columns
X_train_processed = pd.concat([X_train[numerical_cols], X_train_cat_df], axis=1)
X_test_processed = pd.concat([X_test[numerical_cols], X_test_cat_df], axis=1)

print(f"Shape after encoding - Train: {X_train_processed.shape}, Test: {X_test_processed.shape}")
print(f"Feature names: {list(X_train_processed.columns)}")

Shape after encoding - Train: (2977, 14), Test: (1277, 14)
Feature names: ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi', 'gender_Male', 'gender_Other', 'ever_married_Yes', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']


In [13]:
# Apply SMOTE to the encoded training data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_processed, y_train)

print(f"Before SMOTE: {y_train.value_counts()}")
print(f"After SMOTE: {y_resampled.value_counts()}")

Before SMOTE: stroke
False    2814
True      163
Name: count, dtype: int64
After SMOTE: stroke
False    2814
True     2814
Name: count, dtype: int64


In [14]:
# Scale the features (after SMOTE)
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_resampled)
X_test_norm = scaler.transform(X_test_processed)

In [22]:
# Performing GridSearchCV for Lasso-Regression Model
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'saga']  # Optional: tune solver
}

lasso_lr = LogisticRegression(penalty='l1', max_iter=5000, random_state=42)
grid_search_lasso = GridSearchCV(lasso_lr, param_grid, cv=5, scoring='f1')
grid_search_lasso.fit(X_train_norm, y_resampled)

print("Best parameters:", grid_search_lasso.best_params_)
best_lasso = grid_search_lasso.best_estimator_
y_pred_lasso = best_lasso.predict(X_test_norm)

Best parameters: {'C': 0.01, 'solver': 'liblinear'}


In [23]:
# Evaluate Lasso Regression Model
y_pred_lasso = best_lasso.predict(X_test_norm)
y_proba_lasso = best_lasso.predict_proba(X_test_norm)[:, 1]

roc_auc_lasso = roc_auc_score(y_test, y_proba_lasso)
recall_lasso = recall_score(y_test, y_pred_lasso)
f1_lasso = f1_score(y_test, y_pred_lasso)
precision_lasso = precision_score(y_test, y_pred_lasso)

print(f"Lasso ROC-AUC: {roc_auc_lasso:.3f}")
print(f"Lasso Recall: {recall_lasso:.3f}")
print(f"Lasso F1 Score: {f1_lasso:.3f}")
print(f"Lasso Precision: {precision_lasso:.3f}")

Lasso ROC-AUC: 0.840
Lasso Recall: 0.774
Lasso F1 Score: 0.284
Lasso Precision: 0.174


In [24]:
# Performing GridSeearchCV for ElasticNet Model
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

elasticnet_lr = LogisticRegression(penalty='elasticnet', solver='saga', max_iter=5000, random_state=42)
grid_search = GridSearchCV(elasticnet_lr, param_grid, cv=5, scoring='f1')
grid_search.fit(X_train_norm, y_resampled)

print("Best parameters:", grid_search.best_params_)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_norm)

Best parameters: {'C': 0.01, 'l1_ratio': 0.1}


In [25]:
# Evaluate ElasticNet Regression Model
y_pred_elasticnet = best_model.predict(X_test_norm)
y_proba_elasticnet = best_model.predict_proba(X_test_norm)[:, 1]

roc_auc_elasticnet = roc_auc_score(y_test, y_proba_elasticnet)
recall_elasticnet = recall_score(y_test, y_pred_elasticnet)
f1_elasticnet = f1_score(y_test, y_pred_elasticnet)
precision_elasticnet = precision_score(y_test, y_pred_elasticnet)

print(f"Elasticnet ROC-AUC: {roc_auc_elasticnet:.3f}")
print(f"Elasticnet Recall: {recall_elasticnet:.3f}")
print(f"Elasticnet F1 Score: {f1_elasticnet:.3f}")
print(f"Elasticnet Precision: {precision_elasticnet:.3f}")

Elasticnet ROC-AUC: 0.839
Elasticnet Recall: 0.774
Elasticnet F1 Score: 0.293
Elasticnet Precision: 0.181


In [26]:
# GridSearchCV for Random Forest Classifier
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', None]
}

rf = RandomForestClassifier(random_state=42)
grid_search_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='f1')
grid_search_rf.fit(X_train_norm, y_resampled)

print("Best parameters for Random Forest:", grid_search_rf.best_params_)
best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test_norm)

Best parameters for Random Forest: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}


In [27]:
# Predict probabilities and classes using best Random Forest model
y_pred_rf = best_rf.predict(X_test_norm)
y_proba_rf = best_rf.predict_proba(X_test_norm)[:, 1]

In [28]:
# Default threshold metrics
roc_auc_rf = roc_auc_score(y_test, y_proba_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)

print(f"Random Forest ROC-AUC: {roc_auc_rf:.3f}")
print(f"Random Forest Recall: {recall_rf:.3f}")
print(f"Random Forest F1 Score: {f1_rf:.3f}")
print(f"Random Forest Precision: {precision_rf:.3f}")

Random Forest ROC-AUC: 0.758
Random Forest Recall: 0.060
Random Forest F1 Score: 0.085
Random Forest Precision: 0.152


In [29]:
# GridSearchCV for XGBoost Classifier
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

xgb_clf = xgb.XGBClassifier(random_state=42, eval_metric='logloss')
grid_search_xgb = GridSearchCV(xgb_clf, param_grid_xgb, cv=5, scoring='f1', n_jobs=-1)
grid_search_xgb.fit(X_train_norm, y_resampled)

print("Best parameters for XGBoost:", grid_search_xgb.best_params_)
best_xgb = grid_search_xgb.best_estimator_

Best parameters for XGBoost: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 7, 'min_child_weight': 1, 'n_estimators': 200, 'subsample': 0.8}


In [30]:
# Evaluate XGBoost Model
y_pred_xgb = best_xgb.predict(X_test_norm)
y_proba_xgb = best_xgb.predict_proba(X_test_norm)[:, 1]

roc_auc_xgb = roc_auc_score(y_test, y_proba_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)

print(f"XGBoost ROC-AUC: {roc_auc_xgb:.3f}")
print(f"XGBoost Recall: {recall_xgb:.3f}")
print(f"XGBoost F1 Score: {f1_xgb:.3f}")
print(f"XGBoost Precision: {precision_xgb:.3f}")

XGBoost ROC-AUC: 0.760
XGBoost Recall: 0.119
XGBoost F1 Score: 0.167
XGBoost Precision: 0.278


In [31]:
# Compare all models and select the best one
models_comparison = {
    'Lasso': {
        'ROC-AUC': roc_auc_lasso,
        'Recall': recall_lasso,
        'F1': f1_lasso,
        'Precision': precision_lasso,
        'model': best_lasso
    },
    'ElasticNet': {
        'ROC-AUC': roc_auc_elasticnet,
        'Recall': recall_elasticnet,
        'F1': f1_elasticnet,
        'Precision': precision_elasticnet,
        'model': best_model
    },
    'RandomForest': {
        'ROC-AUC': roc_auc_rf,
        'Recall': recall_rf,
        'F1': f1_rf,
        'Precision': precision_rf,
        'model': best_rf
    },
    'XGBoost': {
        'ROC-AUC': roc_auc_xgb,
        'Recall': recall_xgb,
        'F1': f1_xgb,
        'Precision': precision_xgb,
        'model': best_xgb
    }
}

# Display comparison
print("\nModel Comparison:")
print("-" * 80)
for model_name, metrics in models_comparison.items():
    print(f"{model_name}:")
    print(f"  ROC-AUC: {metrics['ROC-AUC']:.3f} | Recall: {metrics['Recall']:.3f} | "
          f"F1: {metrics['F1']:.3f} | Precision: {metrics['Precision']:.3f}")
print("-" * 80)

# Select best model based on recall (prioritize catching stroke cases)
best_model_name = max(models_comparison.keys(), key=lambda x: models_comparison[x]['Recall'])
best_stroke_model = models_comparison[best_model_name]['model']
print(f"\nBest model based on Recall: {best_model_name}")
print(f"Best Recall: {models_comparison[best_model_name]['Recall']:.3f}")


Model Comparison:
--------------------------------------------------------------------------------
Lasso:
  ROC-AUC: 0.840 | Recall: 0.774 | F1: 0.284 | Precision: 0.174
ElasticNet:
  ROC-AUC: 0.839 | Recall: 0.774 | F1: 0.293 | Precision: 0.181
RandomForest:
  ROC-AUC: 0.758 | Recall: 0.060 | F1: 0.085 | Precision: 0.152
XGBoost:
  ROC-AUC: 0.760 | Recall: 0.119 | F1: 0.167 | Precision: 0.278
--------------------------------------------------------------------------------

Best model based on Recall: Lasso
Best Recall: 0.774
